In [3]:
from dotenv import load_dotenv
import vertexai
from langchain_google_vertexai import VertexAIEmbeddings

load_dotenv()
PROJECT_ID = "rag-project-495601"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)

# Khởi tạo mô hình nhúng chính thức đã chọn ở Step 3
embedding_model = VertexAIEmbeddings(model_name="text-embedding-004")
print("Khởi tạo thành công.")

d:\Intern\venv\Lib\site-packages\google\cloud\aiplatform\models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils
d:\Intern\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\XuanDongDev\AppData\Local\Temp\ipykernel_29564\718722219.py:12: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embedding_model = VertexAIEmbeddings(model_name="text-embedding-004")
C:\Users\XuanDongDev\AppData\Local\Temp\ipykernel_29564\718722219.py:12: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An upda

Khởi tạo thành công.


In [4]:
print("🚀 Hệ thống đã sẵn sàng với Gemini Embedding 2 (768-dim).")

🚀 Hệ thống đã sẵn sàng với Gemini Embedding 2 (768-dim).


In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Nạp tệp Markdown bằng TextLoader mặc định (Hỗ trợ tốt UTF-8)
file_path = "../data/outputs/energy_for_rag.md"
loader = TextLoader(file_path, encoding="utf-8")
data = loader.load()

# 2. Cấu hình Splitter (đã chốt ở Step 2)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "]
)

# 3. Thực thi chia nhỏ tài liệu
final_chunks = text_splitter.split_documents(data)

# 4. Gán và tối ưu hóa Metadata cho AusNet Energy
for i, chunk in enumerate(final_chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["source"] = "AusNet_Energy_2025"
    
    # Tự động trích xuất số Trang (Page) từ nội dung đoạn văn bản nếu có
    if "## Page " in chunk.page_content:
        # Lấy dòng chứa nhãn trang
        for line in chunk.page_content.split("\n"):
            if line.startswith("## Page "):
                chunk.metadata["page_label"] = line.replace("## Page ", "").strip()
                break

print(f"Đã xử lý xong {len(final_chunks)} chunks từ tệp Markdown.")
# In thử chunk đầu tiên để nghiệm thu metadata
print("-" * 40)
print(f"Metadata mẫu: {final_chunks[0].metadata}")
print(f"Nội dung mẫu: {final_chunks[0].page_content[:150]}...")

Đã xử lý xong 35 chunks từ tệp Markdown.
----------------------------------------
Metadata mẫu: {'source': 'AusNet_Energy_2025', 'chunk_id': 0, 'page_label': '1'}
Nội dung mẫu: # Document: Household Energy Cost Analysis
# Source: AusNet

## Page 1

Household Energy
Cost Analysis
31 Jan 2025
Assumptions Document

---...


In [6]:
pip install scann

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement scann (from versions: none)
ERROR: No matching distribution found for scann


In [4]:
import os
import json
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Tải biến môi trường từ .env
load_dotenv()

def get_db_uri():
    user = os.getenv("USERNAME")
    password = os.getenv("PASSWORD")
    raw_host = os.getenv("HOST", "")
    
    # Xử lý an toàn nếu HOST chứa luôn PORT
    host = raw_host.split(":")[0] if ":" in raw_host else raw_host
    port = os.getenv("PORT", "5432")
    db_name = os.getenv("DB_NAME", "energy_db")
    
    return f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db_name}"

DB_URI = get_db_uri()
print(f"Đã cấu hình URI kết nối tới: HOST")

# 2. Khởi tạo model E5 Large (1024 chiều)
# Lưu ý: Model này chạy local trên RAM/GPU của bạn
model_name = "intfloat/multilingual-e5-large"
hf_embeddings = HuggingFaceEmbeddings(model_name=model_name)
print(f"Đã tải trọng số {model_name} (1024-dim) thành công.")

Đã cấu hình URI kết nối tới: HOST


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5103.42it/s]


Đã tải trọng số intfloat/multilingual-e5-large (1024-dim) thành công.


In [5]:
# 1. Đọc tệp Markdown
file_path = "../data/outputs/energy_for_rag.md"
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()

# 2. Cấu hình Splitter giữ nguyên cấu trúc Markdown
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, 
    chunk_overlap=120,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "]
)
chunks = text_splitter.split_documents(documents)

print(f"Đã phân mảnh thành công {len(chunks)} chunks từ tệp Markdown.")

Đã phân mảnh thành công 35 chunks từ tệp Markdown.


In [ ]:
    engine = create_engine(DB_URI)

    print(f"Đang bắt đầu quá trình Ingestion vào AlloyDB...")

    with engine.begin() as conn:
        for i, chunk in enumerate(chunks):
            # QUAN TRỌNG: E5-large yêu cầu prefix 'passage: ' khi lưu vào DB
            text_to_embed = f"passage: {chunk.page_content}"
            vector = hf_embeddings.embed_query(text_to_embed)
            
            # Chuyển vector thành chuỗi định dạng postgres [x, y, z...]
            vector_str = f"[{','.join(map(str, vector))}]"
            
            # Gán metadata logic để hỗ trợ Hybrid Search (Lọc theo hộ gia đình)
            household_type = "Electrified + EV" if "EV" in chunk.page_content else "General"
            metadata = {
                "source": "AusNet_Energy_2025", 
                "chunk_id": i,
                "household_type": household_type,
                "state": "VIC" # Giả định dữ liệu AusNet tập trung tại Victoria
            }
            
            # Thực thi INSERT hoặc UPDATE (Idempotent)
            conn.execute(text("""
                INSERT INTO ausnet_energy_poc (id, content, embedding, metadata)
                VALUES (:id, :content, :embedding, :metadata)
                ON CONFLICT (id) DO UPDATE SET 
                    content = EXCLUDED.content,
                    embedding = EXCLUDED.embedding,
                    metadata = EXCLUDED.metadata;
            """), {
                "id": f"ausnet_{i}",
                "content": chunk.page_content,
                "embedding": vector_str,
                "metadata": json.dumps(metadata)
            })
            
            if (i + 1) % 10 == 0:
                print(f"  - Đã nạp {i + 1}/{len(chunks)} chunks...")

    print("\nNạp tri thức thành công! Dữ liệu đã sẵn sàng trong AlloyDB.")

Đang bắt đầu quá trình Ingestion vào AlloyDB...


OperationalError: (psycopg.errors.ConnectionTimeout) connection timeout expired
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# 1. Tạo véc-tơ câu hỏi (E5 yêu cầu prefix 'query: ' cho câu hỏi)
query_text = "query: What are the upfront cost and wiring assumptions for EV?"
query_vector = hf_embeddings.embed_query(query_text)
vector_str = f"[{','.join(map(str, query_vector))}]"

# 2. Truy vấn kết hợp lọc Metadata (Chỉ lấy nhóm Electrified + EV)
sql_search = text("""
    SELECT content, 
           1 - (embedding <=> :v::vector) as similarity_score
    FROM ausnet_energy_poc
    WHERE metadata @> '{"household_type": "Electrified + EV"}'::jsonb
    ORDER BY embedding <-> :v::vector -- Sử dụng <-> để tối ưu cho ScaNN sau này
    LIMIT 3;
""")

print(f"Đang tìm kiếm: '{query_text}'\n")
with engine.connect() as conn:
    results = conn.execute(sql_search, {"v": vector_str})
    for row in results:
        print(f"[Score: {row.similarity_score:.4f}]")
        print(f"Nội dung: {row.content[:200]}...")
        print("-" * 50)